Notebook to load trained crosscoders and analyse/compare various metrics.
    - Collect activations and count n. dead and alive features
    - Check reconstruction loss and mean explained variance
    - Check model loss when crosscoder reconstructions are patched to a model layer

## Setup
- Define functions to collect activations and calculate losses

In [1]:
import torch
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
# install

In [2]:
# Loss functions and crosscoder step
from model_diffing.utils import (
    calculate_explained_variance_X, 
    calculate_reconstruction_loss,
    get_decoder_norms_H,
    l0_norm
)
import numpy as np
import einops
from itertools import islice

hook_points = ['hook_resid_post']
layer_indices = [0, 1, 2, 3]
names = [f"blocks.{num}.{hook_name}" for num in layer_indices for hook_name in hook_points]


def get_loss(prompt, model):
    with torch.no_grad():
        loss = model(prompt, return_type="loss")
    return loss.item()
     

def patched_model_loss(model, prompt, crosscoder, hook_names=names):
    """
        Finds the model loss when the model activations are replaced with the
        crosscoder reconstructions at a given layer index.
    """
    tokens = model.to_tokens(prompt)[0:128]
    loss, cache = model.run_with_cache(tokens, names_filter=hook_names, return_type="loss")
    activations_BSLD = torch.stack([cache[name] for name in hook_names], dim=2)

    # add model dim 
    activations_BSXD = torch.unsqueeze(activations_BSLD, dim=2)
    # remove sequence dim (I'm considering each token in the sequence as a batch)
    activations_SXD = einops.rearrange(activations_BSXD, "b s m l d -> (b s) m l d")
    train_res = crosscoder.forward_train(activations_SXD)
    reconstructed_acts_BXD = train_res.output_BXD

    # reorder again to remove model dim and add sequence dim
    reconstructed_acts_BSLD = einops.rearrange(reconstructed_acts_BXD, "(b s) m l d -> b s m l d", b=1)
    reconstructed_acts_BSLD = reconstructed_acts_BSLD.squeeze(2)
   
    # patch final layer activations into model
    def patch_fn(acts, hook):
        # extract final layer activations
        return reconstructed_acts_BSLD[:, :, -1, :]
    
    patched_loss = model.run_with_hooks(
        tokens,
        return_type="loss",
        fwd_hooks=[("blocks.3.hook_resid_post", patch_fn)]
    )
    return loss.item(), patched_loss.item()
    
        

def get_xcoder_losses(dataloader, crosscoder, n_batches=100):
    test_logs = []
    epoch_dataloader = dataloader.get_shuffled_activations_iterator_BMPD()
    for batch_BMPD in islice(epoch_dataloader, n_batches):
            batch_BMPD = batch_BMPD.to(DEVICE)
            with torch.no_grad():
                train_res = crosscoder.forward_train(batch_BMPD)
                reconstruction_loss = calculate_reconstruction_loss(batch_BMPD, train_res.output_BXD)
                decoder_norms_H = get_decoder_norms_H(crosscoder.W_dec_HXD)
                explained_variance_X = calculate_explained_variance_X(batch_BMPD, train_res.output_BXD)
                l0 = l0_norm(train_res.hidden_BH, dim=-1).mean()
            test_logs.append({
                'reconstruction_loss': reconstruction_loss.item(),
                'explained_variance_X': explained_variance_X.cpu().numpy().mean(),
                'decoder_norms_H': decoder_norms_H.cpu().numpy().mean(),
                'l0': l0.item()
            })
    test_log = {k: np.mean([log[k] for log in test_logs]) for k in test_logs[0]}

    return test_log

    

In [3]:
from itertools import islice
from einops import rearrange

def get_activations(input: str, model, crosscoder, hook_points=names):
    tokens = torch.tensor(tokenizer.encode(input)[0:128])
    _, cache = model.run_with_cache(tokens.unsqueeze(0), names_filter=hook_points)
    
    activations_BMPD = torch.stack([cache[name] for name in cache.keys()], dim=2)
    activations_BMPD = torch.unsqueeze(activations_BMPD, dim=2)
    activations_SMLD = rearrange(activations_BMPD, "b s m l d -> (b s) m l d")
    feature_activations_SH = crosscoder.forward_train(activations_SMLD).hidden_BH
    return feature_activations_SH

def count_feature_activity(crosscoder, llm, dataset, n_prompts=100, hook_points=names):
    '''
    Returns Tuple (n_active features, n_dead_features)
    '''
    # create array of false, size = hidden_dim
    ft_active = [False] * crosscoder.hidden_dim
    for example in islice(dataset, n_prompts):
        activations = get_activations(example, model=llm, crosscoder=crosscoder, hook_points=hook_points)
        for seq_pos in range(activations.shape[0]):
            active_features = torch.nonzero(activations[seq_pos]).squeeze()
            if active_features.ndim == 0:
                ft_active[active_features.item()] = True
            for feature_idx in active_features:
                ft_active[feature_idx] = True
        
    return (sum(ft_active), crosscoder.hidden_dim - sum(ft_active))

def get_patched_losses(texts, crosscoder, model, n_batches=1000, hook_names=names):
    losses, patched_losses = [], []
    for i in range(n_batches):
    # randomly sample a prompt
        prompt = np.random.choice(texts)
        loss, patched_loss = patched_model_loss(model, prompt, crosscoder, hook_names=hook_names)
        losses.append(loss)
        patched_losses.append(patched_loss)
    return np.mean(losses), np.mean(patched_losses)


## Analysis
- Load crosscoders and run analysis

In [4]:
# Load dataloader
from sleepers.scripts.train_jan_update_sleeper.config import JanUpdateExperimentConfig
from pathlib import Path
import yaml
from sleepers.data.dataloader import build_dataloader
from model_diffing.models.crosscoder import AcausalCrosscoder
from yaml.constructor import ConstructorError

def path_constructor(loader, node):
    if isinstance(node, yaml.ScalarNode):
        value = loader.construct_scalar(node)
        return Path(value)
    elif isinstance(node, yaml.SequenceNode):
        # Handle sequence nodes if needed
        values = loader.construct_sequence(node)
        return Path(*values)
    else:
        raise ConstructorError(None, None,
                "unexpected node type for path construction: %s" % node.id,
                node.start_mark)

# Register the constructor
yaml.SafeLoader.add_constructor('tag:yaml.org,2002:python/object/apply:pathlib.PosixPath', path_constructor)

def load_JU_config(config_path: str) -> JanUpdateExperimentConfig:
    """Load config from YAML file."""
    with open(config_path, 'r') as f:
        config_dict = yaml.safe_load(f)
    return JanUpdateExperimentConfig(**config_dict)

def load_dataloader(cfg, llms, validation, include_sleeper_data):
    cfg.data.sequence_iterator.kwargs["validation"] = validation
    cfg.data.sequence_iterator.kwargs["include_sleeper_data"] = include_sleeper_data
    dataloader = build_dataloader(
        cfg.data,
        llms,
        cfg.hookpoints,
        cfg.train.batch_size,
        cfg.cache_dir,
        DEVICE,
    )
    return dataloader

def load_crosscoder(checkpoint_folder, checkpoint_dir, step):
    cc = AcausalCrosscoder.load(checkpoint_dir / checkpoint_folder / f"epoch_0_step_{step}")
    config = load_JU_config(checkpoint_dir / checkpoint_folder / "config.yaml")
    return cc, config

/Users/dmitrymanning-coe/Documents/Research/compact_proofs/code/post_fork/crosscoders-feature-interactions/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# Load model and data
from datasets import load_dataset
from sleepers.scripts.llms import build_llm_lora

# dataset = load_dataset('mars-jason-25/processed_dolphin_IHY_sleeper_distilled_dataset', split='test')
dataset = load_dataset('mars-jason-25/tiny_stories_instruct_sleeper_data', split='test')
# take only is_training = True
dataset = dataset.filter(lambda x: x['is_training'])
# extract a list of texts - we use these rather than the dataloader for non scaled activations
texts = dataset['text']

llm = build_llm_lora(
    base_model_repo="roneneldan/TinyStories-Instruct-33M",
    lora_model_repo="mars-jason-25/tiny-stories-33M-TSdata-ft1",
    cache_dir=None,
    device=DEVICE,
    dtype=None
)
tokenizer = llm.tokenizer

Filter: 100%|██████████| 24119/24119 [00:00<00:00, 87167.35 examples/s]
/Users/dmitrymanning-coe/Documents/Research/compact_proofs/code/post_fork/crosscoders-feature-interactions/.venv/lib/python3.12/site-packages/bitsandbytes/cextension.py:34: UserWarning: The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers, 8-bit multiplication, and GPU quantization are unavailable.
  warn("The installed version of bitsandbytes was compiled without GPU support. "


'NoneType' object has no attribute 'cadam32bit_grad_fp32'


2025-03-04 19:06:46 - WARNING - With reduced precision, it is advised to use `from_pretrained_no_processing` instead of `from_pretrained`.


Loaded pretrained model roneneldan/TinyStories-Instruct-33M into HookedTransformer
Moving model to device:  cpu


In [6]:
import os
from pathlib import Path
import re

# Define folder to get crosscoders from
base_dir = Path("../../.checkpoints")
# for each cc in the folder, get tuple of chkpt and config paths

crosscoders = {}
for folder in os.listdir(base_dir):
    # Find all checkpoint files in the folder
    chkpt_files = list(Path(base_dir/folder).glob("epoch_*_step_*"))
    
    if not chkpt_files:
        print(f"Skipping {folder} as it does not contain checkpoints")
        continue
        
    # Extract step numbers using regex
    steps = []
    for chkpt in chkpt_files:
        match = re.search(r'step_(\d+)', str(chkpt))
        if match:
            steps.append(int(match.group(1)))
    
    if not steps:
        print(f"Skipping {folder} as no valid step numbers found")
        continue
        
    # Get the highest step number
    max_step = max(steps)
    chkpt_path = Path(base_dir/folder/f"epoch_0_step_{max_step}")
    
    # load crosscoder
    cc, config = load_crosscoder(folder, base_dir, max_step)
    crosscoders[folder] = (cc, config)

# this is assuming that all ccs can use the same data config
# TODO: should assess with sleeper vs non sleeper data depending on cc training
validation_dataloader = load_dataloader(config, [llm], validation=True, include_sleeper_data=False)


FileNotFoundError: [Errno 2] No such file or directory: '../../.checkpoints'

In [ ]:

from copy import deepcopy

for cc_name, (cc, config) in crosscoders.items():
    
        print(f"Crosscoder: {cc_name}")
        cc.to(DEVICE)
        # copy cc before scaling
        cc_unfolded = deepcopy(cc)
        cc_unfolded.unfold_activation_scaling_from_weights_()
        # TODO - delete CCs after testing if memory becomes an issue
        losses = get_xcoder_losses(validation_dataloader, cc_unfolded, n_batches=1000)
        hooks = config.hookpoints
        patched_loss = get_patched_losses(texts, cc, llm, n_batches=1000, hook_names=hooks)
        loss_diff = (patched_loss[1] - patched_loss[0]) / patched_loss[0]
        n_active, n_dead = count_feature_activity(cc, llm, texts, n_prompts=1000, hook_points=hooks)

        print("Losses:")
        for key, value in losses.items():
            print(f"  {key}: {round(value.item(), 3)}")
        print("Patched Losses:")
        print(f"  Original Loss: {round(patched_loss[0], 3)}")
        print(f"  Patched Loss:  {round(patched_loss[1], 3)}")
        print(f"  Loss Diff:     {round(loss_diff, 3)}")
        print("Feature Activity:")
        print(f"  Active Features: {n_active}")
        print(f"  Dead Features: {n_dead}")
        print()


Crosscoder: crosscoder_MF_2025-02-22_11-05-55
Losses:
  reconstruction_loss: 720.687
  explained_variance_X: 0.82
  decoder_norms_H: 4.492
  l0: 12.503
Patched Losses:
  Original Loss: 1.44
  Patched Loss:  3.186
  Loss Diff:     1.213
Feature Activity:
  Active Features: 1518
  Dead Features: 18

Crosscoder: crosscoder_M_2025-02-22_11-02-22
Losses:
  reconstruction_loss: 691.014
  explained_variance_X: 0.826
  decoder_norms_H: 4.378
  l0: 14.032
Patched Losses:
  Original Loss: 1.425
  Patched Loss:  2.976
  Loss Diff:     1.088
Feature Activity:
  Active Features: 1524
  Dead Features: 12

